In [0]:
%run ../00-common/01.environment-config

In [0]:
from pyspark.sql.functions import col, lit, when

In [0]:
# Define table names using environment variables
silver_results_table = f"{catalog_name}.{silver_schema}.results"
silver_sprints_table = f"{catalog_name}.{silver_schema}.sprints"
gold_fact_session_results_table = f"{catalog_name}.{gold_schema}.fact_session_results"

# Read silver results table
results_df = spark.table(silver_results_table)

# Read silver sprints table
sprints_df = spark.table(silver_sprints_table)

In [0]:
# Drop unnecessary columns from results table
results_cleaned = results_df.drop(
    "race_name", "race_date", "ingestion_timestamp", "source_file_path"
)

# Drop unnecessary columns from sprints table
sprints_cleaned = sprints_df.drop(
    "race_name", "race_date", "ingestion_timestamp", "source_file_path"
)

# Add session_type column to results (race)
results_with_session_type = results_cleaned.withColumn("session_type", lit("RACE"))

# Add session_type column to sprints (sprint)
sprints_with_session_type = sprints_cleaned.withColumn("session_type", lit("SPRINT"))

# Union both tables
combined_df = results_with_session_type.unionByName(sprints_with_session_type)

In [0]:
# Derive additional analytical columns
fact_session_results_df = (
    combined_df
    # is_win: driver won the race/sprint (final_position = 1)
    .withColumn("is_win", when(col("final_position") == 1, True).otherwise(False))
    # is_podium: driver scored podium (final_position in 1, 2, 3)
    .withColumn("is_podium", when(col("final_position").isin([1, 2, 3]), True).otherwise(False))
    # has_points: driver scored points (points > 0)
    .withColumn("has_points", when(col("points") > 0, True).otherwise(False))
)

In [0]:
# Write transformed data to gold layer
(
    fact_session_results_df.write
    .mode("overwrite")
    .format("delta")
    .saveAsTable(gold_fact_session_results_table)
)

print(f"✓ Session results fact table successfully written to {gold_fact_session_results_table}")

In [0]:
# Preview the fact table
# spark.table(gold_fact_session_results_table).limit(10).display()